# Sources

**Dataset:** https://graph.openaire.eu/docs/

**API:** https://api.openaire.eu/graph/swagger-ui/index.html#/Research%20products/search

In [18]:
import requests
import json
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
import sys
import os

# `organizations`
Since our focus is the *University of Pisa*, this filter is the most direct.

First of all, all entries containing "Pisa" are retrieved.

In [27]:
url = "https://api.openaire.eu/graph/v1/organizations?search=Pisa&page=1&pageSize=100&sortBy=relevance%20DESC"
headers = {"accept": "application/json"}

try:
    response = requests.get(url, headers=headers, timeout=30)
    if response.ok:
        data_Pisa = response.json()
        print(f"[RESPONSE]\tLoaded in 'data_Pisa'\n  [N. RECORDS]\t{len(data_Pisa.get('results', []))}\n  [KEYS]\t{data_Pisa.keys()}")
    else:
        data_Pisa = None
        print(f"Error: {response.status_code} - {response.reason}")
except requests.exceptions.RequestException as e:
    data_Pisa = None
    print(f"Network error: {e}")

[RESPONSE]	Loaded in 'data_Pisa'
  [N. RECORDS]	50
  [KEYS]	dict_keys(['header', 'results'])


In [30]:
# Print unique structure of 'results'
if "results" in data_Pisa:
    if data_Pisa["results"]:
        print("[RESULTS' STRUCTURE]")
        for key in data_Pisa["results"][0].keys():
            print(f"  - {key}")

        first_keys = set(data_Pisa["results"][0].keys())
        all_same = all(set(result.keys()) == first_keys for result in data_Pisa["results"])
        print(f"\n[ALL RESULTS SAME STRUCTURE] {all_same}")

        if not all_same:
            print("\n⚠️ Different structures found in 'results':")
            key_structures = [tuple(sorted(result.keys())) for result in data_Pisa["results"]]
            from collections import Counter
            counter = Counter(key_structures)
            for i, (keys, count) in enumerate(counter.items(), 1):
                print(f"  Structure {i}:")
                for k in keys:
                    print(f"    - {k}")
                print(f"    [Count: {count}]\n")

[RESULTS' STRUCTURE]
  - legalShortName
  - legalName
  - websiteUrl
  - alternativeNames
  - country
  - id
  - pids

[ALL RESULTS SAME STRUCTURE] True


In [32]:
# Print unique legalName values in copy-paste format for array creation
if "results" in data_Pisa and data_Pisa["results"]:
    unique_legal_names = {entry.get("legalName") for entry in data_Pisa["results"] if entry.get("legalName")}
    
    print(f"[legalName unique] {len(unique_legal_names)}")
    print("legal_names = [")
    for legal_name in sorted(unique_legal_names):
        print(f'    "{legal_name}",')
    print("]")
else:
    print("No results found.")

[legalName unique] 47
legal_names = [
    "Associazione Italiana Persone Down Onlus Pisa",
    "Associazione nazionale insegnanti scienze naturali Pisa",
    "Azienda Ospedaliero Universitaria Pisana",
    "Ball State University",
    "British School Pisa srl",
    "COMUNE DI PISA",
    "Consorzio Pisa Ricerche",
    "Department of Chemistry and Industrial Chemistry - Università di Pisa",
    "Department of Civilization and forms of knowledge - University of Pisa",
    "Dipartimento di Fisica Università degli Studi di Pisa",
    "Dipartimento di Fisica, Università di Pisa and INFN Sezione di Pisa",
    "Dipartimento di Studi Italianistici - Università di Pisa",
    "Dipartimento di scienze economiche - Università di Pisa",
    "Expertisecentrum Nederlands",
    "Fonctions Optiques pour les Technologies de l’information",
    "Fondazione Teatro di Pisa",
    "INGV - Pisa",
    "IPSAR G. MATTEOTTI PISA",
    "Institut Parisien de Chimie Moléculaire",
    "Institut de Recherche en Informa

In [9]:
legal_names = [
    "Azienda Ospedaliero Universitaria Pisana",
    "Department of Chemistry and Industrial Chemistry - Università di Pisa",
    "Department of Civilization and forms of knowledge - University of Pisa",
    "Dipartimento di Fisica Università degli Studi di Pisa",
    "Dipartimento di Fisica, Università di Pisa and INFN Sezione di Pisa",
    "Dipartimento di Studi Italianistici - Università di Pisa",
    "Dipartimento di scienze economiche - Università di Pisa",
    "Research Center E. Piaggio - University of Pisa",
    "University of Pisa",
    "University of Pisa, Department of Clinical and Experimental Medecine, Division of Pharmacology, Pisa, Italy",
    "Università degli Studi di Pisa - Dipartimento die Matematica",
]

In [ ]:
# Filter results to create new object with only entries in legal_names
if "results" in data_Pisa and data_Pisa["results"]:
    # Create filtered dataset
    filtered_data_Pisa = {
        "results": [
            entry for entry in data_Pisa["results"] 
            if entry.get("legalName") in legal_names
        ]
    }
    
    # Copy other keys from original data if they exist
    for key in data_Pisa.keys():
        if key != "results":
            filtered_data_Pisa[key] = data_Pisa[key]
    
    print(f"[ORIGINAL] {len(data_Pisa['results'])}")
    print(f"[FILTERED] {len(filtered_data_Pisa['results'])}")
    print(f"[REMOVED] {len(data_Pisa['results']) - len(filtered_data_Pisa['results'])}")

    # Show which legal names were found
    found_names = {entry.get("legalName") for entry in filtered_data_Pisa["results"]}
    missing_names = set(legal_names) - found_names

    print(f"\n[FOUND] {len(found_names)}")
    print(f"[MISSING] {len(missing_names)}")

    if missing_names:
        print("\nMissing names:")
        for name in sorted(missing_names):
            print(f"  - {name}")
else:
    print("No results found to filter.")

Original total: 50
After filtering with legal_names: 11
Removed: 39

Found legal names: 11
Missing legal names: 0


In [ ]:
# Create output directory if it doesn't exist
output_dir = "./output"
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

# Save only the results array to JSON file
output_file = os.path.join(output_dir, "pisa_organizations.json")

if 'filtered_data_Pisa' in locals() and 'results' in filtered_data_Pisa:
    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(filtered_data_Pisa['results'], f, ensure_ascii=False, indent=2)

    print(f"[SUCCESS] saved to: {output_file} - contains {len(filtered_data_Pisa['results'])} organizations")
else:
    print("[ERROR] No filtered results found to save.")

Results data saved to: ./output/pisa_organizations.json
File contains 11 organizations


In [ ]:
import requests
import json
import os
import time
from typing import List, Dict

class ResearchProductsFetcher:
    def __init__(self, output_dir="research_products_data"):
        self.api_url = "https://api.openaire.eu/graph/v2/researchProducts"
        self.output_dir = output_dir
        self.headers = {"accept": "application/json"}
        os.makedirs(output_dir, exist_ok=True)

    def fetch_all(self, organizations: List[Dict]) -> Dict:
        products = {}
        for org in organizations:
            org_id = org.get("id")
            org_name = org.get("legalShortName", org.get("legalName", "Unknown"))
            if not org_id:
                print(f"[ORG.] No ID: {org_name}")
                continue
            print(f"\nProcessing: {org_name} (ID: {org_id})")
            for prod in self._fetch_by_cursor(org_id):
                prod_id = prod.get("id")
                if not prod_id:
                    continue
                if prod_id not in products:
                    prod["affiliated_organizations"] = [{"id": org_id, "name": org_name}]
                    products[prod_id] = prod
                else:
                    products[prod_id].setdefault("affiliated_organizations", []).append({"id": org_id, "name": org_name})
            print(f"[FOUND] {len(products)} unique products so far")
            time.sleep(0.5)
        return products

    def _fetch_by_cursor(self, org_id: str) -> List[Dict]:
        all_products = []
        cursor = "*"
        page_size = 100
        while cursor:
            params = {
                "relOrganizationId": org_id,
                "cursor": cursor,
                "pageSize": page_size,
                "sortBy": "publicationDate DESC"
            }
            try:
                resp = requests.get(self.api_url, params=params, headers=self.headers, timeout=30)
                if resp.status_code != 200:
                    print(f"  ⚠️  Error {resp.status_code}")
                    break
                data = resp.json()
                results = data.get("results", [])
                if not results:
                    break
                all_products.extend(results)
                cursor = data.get("header", {}).get("nextCursor")
                print(f"  [FETCHED] {len(all_products)} products")
                time.sleep(0.3 if len(all_products) < 5000 else 1)
            except Exception as e:
                print(f"  [Error] {e}")
                break
        return all_products

    def save(self, products: Dict, prefix="research_products"):
        products_list = list(products.values())
        main_file = os.path.join(self.output_dir, f"{prefix}_complete.json")
        with open(main_file, "w", encoding="utf-8") as f:
            json.dump(products_list, f, indent=2, ensure_ascii=False)
        print(f"\n[SUCCESS] Saved {len(products_list)} products to: {main_file}")

        compact_file = os.path.join(self.output_dir, f"{prefix}_compact.json")
        compact = []
        for prod in products_list:
            item = {
                "id": prod.get("id"),
                "mainTitle": prod.get("mainTitle"),
                "publicationDate": prod.get("publicationDate"),
                "type": prod.get("type"),
                "doi": next((pid.get("value") for pid in prod.get("pid", []) if pid.get("scheme") == "doi"), None),
                "affiliated_organizations": prod.get("affiliated_organizations", [])
            }
            compact.append(item)
        with open(compact_file, "w", encoding="utf-8") as f:
            json.dump(compact, f, indent=2, ensure_ascii=False)
        print(f"[SUCCESS] Saved compact version to: {compact_file}")

        self._save_stats(products_list)

    def _save_stats(self, products: List[Dict]):
        stats = {
            "total": len(products),
            "by_type": {},
            "by_year": {},
            "with_doi": 0,
            "organizations": set()
        }
        for prod in products:
            t = prod.get("type", "unknown")
            stats["by_type"][t] = stats["by_type"].get(t, 0) + 1
            if prod.get("publicationDate"):
                y = prod["publicationDate"][:4]
                stats["by_year"][y] = stats["by_year"].get(y, 0) + 1
            if any(pid.get("scheme") == "doi" for pid in prod.get("pid", [])):
                stats["with_doi"] += 1
            for org in prod.get("affiliated_organizations", []):
                stats["organizations"].add(org["name"])
        stats["organizations"] = list(stats["organizations"])
        stats_file = os.path.join(self.output_dir, "statistics.json")
        with open(stats_file, "w", encoding="utf-8") as f:
            json.dump(stats, f, indent=2, ensure_ascii=False)
        print(f"[SUCCESS] Stats saved to: {stats_file}")
        print("\nSUMMARY:")
        print(f"     [Total] products: {stats['total']}")
        print(f"     [With DOI] products: {stats['with_doi']}")
        print(f"     [Organizations] count: {len(stats['organizations'])}")
        print(f"     [Types] distribution: {stats['by_type']}")

In [17]:
with open("./output/pisa_organizations.json", 'r') as f:
    organizations = json.load(f)

fetcher = ResearchProductsFetcher()
products = fetcher.fetch_all(organizations)
fetcher.save(products)


Processing: Department of Civilization and forms of knowledge - University of Pisa (ID: openorgs____::c186a10ae4acdf58596bacc91c2cc620)
  Fetched 60 products
✓ Found 60 unique products so far

Processing: Dipartimento di Chimica e Chimica Industriale (ID: openorgs____::0462274ad38f4d94dbe86d320960850e)
  Fetched 49 products
✓ Found 109 unique products so far

Processing: Dipartimento di Studi Italianistici - Università di Pisa (ID: openorgs____::6cac13134787940043685a42cc596d23)
✓ Found 109 unique products so far

Processing: Dipartimento di Fisica, Università di Pisa and INFN Sezione di Pisa (ID: pending_org_::ab0cf7bec4e87cabe665fa48b5ae9a79)
  Fetched 57 products
✓ Found 166 unique products so far

Processing: Dipartimento di scienze economiche - Università di Pisa (ID: openorgs____::5849bf7ec1a1fff084145719ffa11ca6)
  Fetched 2 products
✓ Found 168 unique products so far

Processing: University of Pisa, Department of Clinical and Experimental Medecine, Division of Pharmacology, Pi

KeyboardInterrupt: 